# 3. How much do the results depend on the simulator's assumptions?

A strategy's numbers in a simulator rest on two assumptions the data can't settle: how long orders and data
take to travel (latency) and where an order sits in the queue (the queue model). This notebook sweeps both for
the Avellaneda-Stoikov market maker and shows how fills, PnL and markouts move.

The experiment is defined here in code, the same as a YAML file for `python -m research.run`; runs go to the
store, so running the notebook again takes them from there. On the committed one-minute slice it takes a few
seconds; set `DATASET` and `WINDOW` for the recorded day.

In [ ]:
from pathlib import Path

import numpy as np
import polars as pl
from IPython.display import display

import blocksprint_sim as bs
from research.analytics import report

# The repository root, wherever the notebook is opened from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research" / "analytics").is_dir())

import os

from research.experiments.runner import run_experiment
from research.experiments.schema import Experiment

RESULTS = os.environ.get("BS_RESULTS", str(ROOT / "results"))
WORKERS = int(os.environ.get("BS_WORKERS", "2"))
DATASET, WINDOW = str(ROOT / "tests/sim/data/btcusdt_2026-11-05_0200_1m.bse"), None
# DATASET = str(ROOT / "data/packed/binance_spot_btc_eth_1110")
# WINDOW = {"from": "2026-11-10T02:00:00Z", "to": "2026-11-10T03:00:00Z", "warmup_s": 60}

exp = Experiment.model_validate({
    "name": "notebook_sensitivity", "dataset": DATASET, "symbol": "BTCUSDT", "results": RESULTS, "seeds": [1],
    **({"window": WINDOW} if WINDOW else {}),
    "fees": {"maker_rate": "0.001", "taker_rate": "0.001"},
    "strategy": {"python": "research.strategies.avellaneda_stoikov:AvellanedaStoikov",
                 "params": {"max_position": "0.005", "refresh_ms": 250}},
    "sweep": {"params": {
        "latency_preset": ["zero", "binance_spot_tokyo_vps", "binance_spot_home_pc"],
        "queue_model": [{"model": "cancel_from_front"}, {"model": "pro_rata"}, {"model": "prob_power", "n": 2},
                        {"model": "cancel_from_back"}],
    }},
})
print(f"{len(exp.plan())} runs")

In [ ]:
result = run_experiment(exp, workers=WORKERS, progress=None)
print(result.counts())

## The runs and their headline numbers

`report.metrics` gives each run's PnL and its parts, fills, fill ratio, the 5 s markout and max drawdown (the
numbers of the comparison page).

In [ ]:
rows = []
for rec in result.runs:
    m = report.metrics(result.load(rec))
    rows.append({"latency": rec.params["latency_preset"], "queue": rec.params["queue_model"]["model"], **m})
table = pl.DataFrame(rows)
table.select("latency", "queue", "pnl", "fills", "fill_ratio", "markout_5s_bps", "max_drawdown")

## Fills and PnL across the grid

Rows: queue models from the most optimistic for a passive order (`cancel_from_front`: cancellations ahead of it
move it up) to the most pessimistic (`cancel_from_back`: only trades move it). Columns: latency from none to a
home PC's.

In [ ]:
lat = ["zero", "binance_spot_tokyo_vps", "binance_spot_home_pc"]
qms = ["cancel_from_front", "pro_rata", "prob_power", "cancel_from_back"]


def grid(col):
    g = np.full((len(qms), len(lat)), np.nan)
    for r in table.iter_rows(named=True):
        g[qms.index(r["queue"]), lat.index(r["latency"])] = r[col]
    return g


short = ["zero", "Tokyo VPS", "home PC"]
html = (report.card("Fills", report.heatmap("latency", short, "queue model", qms, grid("fills"), title="fills", digits=0))
        + report.card("PnL (USDT)", report.heatmap("latency", short, "queue model", qms, grid("pnl"), title="pnl",
                                                   diverging=True))
        + report.card("Markout at 5 s (bps)", report.heatmap("latency", short, "queue model", qms,
                                                             grid("markout_5s_bps"), title="markout", digits=3,
                                                             diverging=True)))
display(report.frame(html, 1100))

## How far apart are the answers?

For each number: the spread between the most optimistic and the most pessimistic assumption, relative to the
middle case (Tokyo latency, pro-rata queue). A strategy whose verdict flips across this range can't be judged
from the simulator alone: it needs the paper-trading parity checks of Phase 7.

In [ ]:
mid = table.filter((pl.col("latency") == "binance_spot_tokyo_vps") & (pl.col("queue") == "pro_rata")).row(0, named=True)
pl.DataFrame([{"metric": c, "min": table[c].min(), "middle case": mid[c], "max": table[c].max(),
               "range / middle": (table[c].max() - table[c].min()) / abs(mid[c]) if mid[c] else None}
              for c in ("fills", "pnl", "fill_ratio", "markout_5s_bps", "max_drawdown")])

In [ ]:
path = report.write_reports(result, exp, progress=None)
print("comparison report and tearsheets:", path)